<a href="https://colab.research.google.com/github/carrisian/del-big-data-al-modelo-predictivo/blob/main/prediccion1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
import os

# --- FASE 1: CARGA Y EDA ---
df = pd.read_parquet("Murcia_Dataset_Completo_Global_3H.parquet")
print(f"Dataset cargado: {df.shape[0]} registros, {df.shape[1]} variables.")

# 1. Correlación Multivariante
plt.figure(figsize=(10,8))
corr = df[['PM10', 'NO2', 'Ozono', 'Temp', 'Hum', 'Viento_Vel', 'Radiacion']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Relación entre variables atmosféricas y contaminantes")
plt.savefig("correlacion_variables.png") # Guardamos para el TFM
plt.show()

# 2. Perfil estacional
df['time'] = pd.to_datetime(df['time']) # Asegurar formato fecha
estacion_ejemplo = df[df['Estacion'] == 'Murcia Capital']
estacion_ejemplo.set_index('time')[['Ozono', 'Temp']].resample('ME').mean().plot(secondary_y='Temp')
plt.title("Tendencia mensual: Ozono vs Temperatura (Murcia Capital)")
plt.savefig("tendencia_estacional.png") # Guardamos para el TFM
plt.show()

# --- FASE 2: ENTRENAMIENTO LSTM ---
features = ['PM10', 'Temp', 'Hum', 'Viento_Vel', 'Radiacion']
data = df[features].values
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data)

def create_sequences(data, seq_length=8):
    x, y = [], []
    for i in range(len(data) - seq_length):
        x.append(data[i:(i + seq_length)])
        y.append(data[i + seq_length, 0])
    return np.array(x), np.array(y)

X, y = create_sequences(data_scaled)

# Definir modelo
model = tf.keras.Sequential([
    tf.keras.layers.LSTM(50, return_sequences=True, input_shape=(8, 5)),
    tf.keras.layers.LSTM(50),
    tf.keras.layers.Dense(1)
])

model.compile(optimizer='adam', loss='mse')

# Entrenamiento
history = model.fit(X, y, epochs=20, batch_size=32, validation_split=0.1, verbose=1)

# Visualización del progreso
plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.legend()
plt.title("Convergencia del modelo de IA")
plt.savefig("convergencia_modelo.png") # Guardamos para el TFM
plt.show()

# --- FASE 3: EXPORTACIÓN DE RESULTADOS ---
# Guardar el modelo
model.save("modelo_calidad_aire_murcia.keras")

print("\n--- 🚀 PROCESO COMPLETADO ---")
print("✅ Archivo del modelo: 'modelo_calidad_aire_murcia.keras'")
print("✅ Gráficos exportados: 'correlacion_variables.png', 'tendencia_estacional.png', 'convergencia_modelo.png'")
print("Descárgalos desde la pestaña de Archivos (icono de carpeta a la izquierda) en Colab.")

FileNotFoundError: [Errno 2] No such file or directory: 'Murcia_Dataset_Completo_Global_3H.parquet'